# Define Canonical Model

In [1]:
from dataclasses import dataclass,field
from datetime import datetime, timezone
from typing import Any

@dataclass(frozen=True, slots=True)
class SourceFile:
    source_uri:str
    filename:str
    content:bytes

@dataclass(frozen=True, slots=True)
class DocumentRecord:
    document_id:str
    source_uri: str
    filename: str
    active_version_id:str
    metadata: dict[str, Any] = field(default_factory=dict)

@dataclass(frozen=True, slots=True)
class DocumentVersion:
    version_id: str
    document_id: str
    version_number: int
    content_hash: str
    size_bytes: int
    created_at: datetime


## Implement the content hasher

In [2]:
import hashlib

class ContentHasher:
    """Computes fingerprints for raw document content."""

    @staticmethod
    def sha256(content: bytes)-> str:
        return hashlib.sha256(content).hexdigest()


In [3]:
from pathlib import Path

import os

class InvalidSourceError(ValueError):
    pass


class FileTooLargeError(InvalidSourceError):
    pass


class UnsupportedFileTypeError(InvalidSourceError):
    pass

class TextFileReader:
    def __init__(self, max_size_bytes: int = 5 * 1024 * 1024):
        if max_size_bytes <= 0:
            raise ValueError("max_size_bytes must be positive")
        self.max_size_bytes = max_size_bytes

    def read(self, path: str | Path) -> SourceFile:
        path = Path(path).resolve(strict=True)

        if not path.is_file():
            raise InvalidSourceError("Source is not a regular file")

        if path.suffix.lower() != ".txt":
            raise UnsupportedFileTypeError(path.name)

        with path.open("rb") as handle:
            if os.fstat(handle.fileno()).st_size > self.max_size_bytes:
                raise FileTooLargeError(path.name)

            content = handle.read(self.max_size_bytes + 1)

        if len(content) > self.max_size_bytes:
            raise FileTooLargeError(path.name)

        try:
            content.decode("utf-8")
        except UnicodeDecodeError as exc:
            raise InvalidSourceError("Expected UTF-8 text") from exc

        return SourceFile(
            source_uri=path.as_uri(),
            filename=path.name,
            content=content,
        )


def sha256(content: bytes) -> str:
    return hashlib.sha256(content).hexdigest()


In [4]:
sample_dir = Path("data/samples")
sample_dir.mkdir(parents=True, exist_ok=True)

sample_path = sample_dir / "architecture.txt"

sample_path.write_text("""
    KnowledgeForge is a document retrieval system.
    It uses Python for ingestion.
    Documents are chunked and embedded.
    Chroma stores the vectors.
    Ollama Qwen generates answers using retrieved context.
""".strip(), encoding="utf-8")

reader = TextFileReader()
source = reader.read(sample_path)


print("Filename:", source.filename)
print("Size:", len(source.content))
print("SHA-256:", sha256(source.content))


Filename: architecture.txt
Size: 210
SHA-256: 0b6ecab984f3c12cada643d24825629e63e44e7b81b210a94f2c4566e8da22bb


## Test the reader

In [5]:
assert source.filename == "architecture.txt"
assert len(sha256(source.content)) == 64
assert sha256(source.content) == sha256(source.content)

# Empty files are permitted by our registration policy.
empty_path = sample_dir / "empty.txt"
empty_path.write_bytes(b"")

assert reader.read(empty_path).content == b""

In [6]:
from typing import Protocol
from dataclasses import replace

class DocumentRegistry(Protocol):

    def get(self, document_id: str) -> DocumentRecord | None:
        ...

    def get_active_version(self, document_id: str) -> DocumentVersion | None:
        ...

    def list_versions(self, document_id: str) -> list[DocumentVersion]:
        ...

    def create(
        self,
        document: DocumentRecord,
        version: DocumentVersion,
    ) -> None:
        ...

    def append_version(
        self,
        document_id: str,
        version: DocumentVersion,
    ) -> None:
        ...
class InMemoryDocumentRegistry:

    def __init__(self):
        self._documents = {}
        self._versions = {}

    def get(self, document_id: str) -> DocumentRecord | None:
        record = self._documents.get(document_id)
        return (
            replace(record, metadata=dict(record.metadata))
            if record is not None else None
        )
    def get_active_version(self, document_id):
         document = self.get(document_id)
         if document is None:
            raise KeyError(f"Unknown document: {document_id}")
         return next(
            v for v in self._versions[document_id]
            if v.version_id == document.active_version_id
        )

    def list_versions(self, document_id):
        if document_id not in self._documents:
            raise KeyError(document_id)
        return list(self._versions[document_id])

    def create(self, document, version):
        if document.document_id in self._documents:
            raise ValueError("Document already exists")

        if version.document_id != document.document_id:
            raise ValueError("Document/version mismatch")

        if version.version_number != 1:
            raise ValueError("Initial version must be 1")

        if document.active_version_id != version.version_id:
            raise ValueError("Active version mismatch")

        self._documents[document.document_id] = replace(
            document, metadata=dict(document.metadata)
        )
        self._versions[document.document_id] = [version]

    def append_version(self, document_id, version):
        document = self.get(document_id)
        if document is None:
            raise KeyError(document_id)

        previous = self._versions[document_id][-1]

        if version.document_id != document_id:
            raise ValueError("Document/version mismatch")

        if version.version_number != previous.version_number + 1:
            raise ValueError("Invalid version number")

        if version.version_id == previous.version_id:
            raise ValueError("Duplicate version ID")

        if version.content_hash == previous.content_hash:
            raise ValueError("Content unchanged")

        self._versions[document_id].append(version)
        self._documents[document_id] = replace(
            document,
            active_version_id=version.version_id,
        )

## Registration Actions

In [7]:
from enum import Enum


class RegistrationAction(str, Enum):
    CREATE = "create"
    UPDATE = "update"
    UNCHANGED = "unchanged"


@dataclass(frozen=True)
class RegistrationResult:
    action: RegistrationAction
    document_id: str
    version_id: str
    reason: str

In [8]:
import uuid
class DocumentRegistrationService:
    def __init__(self, registry: DocumentRegistry):
        self.registry = registry

    def create(self, source: SourceFile) -> RegistrationResult:
        document_id = str(uuid.uuid4())
        version_id = str(uuid.uuid4())

        version = DocumentVersion(
            version_id=version_id,
            document_id=document_id,
            version_number=1,
            content_hash=sha256(source.content),
            size_bytes=len(source.content),
            created_at=datetime.now(timezone.utc),
        )

        document = DocumentRecord(
            document_id=document_id,
            source_uri=source.source_uri,
            filename=source.filename,
            active_version_id=version_id,
        )

        self.registry.create(document, version)

        return RegistrationResult(
            RegistrationAction.CREATE,
            document_id,
            version_id,
            "New logical document",
        )

    def update(
        self,
        document_id: str,
        source: SourceFile,
    ) -> RegistrationResult:

        document = self.registry.get(document_id)
        if document is None:
            raise KeyError(document_id)

        previous = self.registry.get_active_version(document_id)
        new_hash = sha256(source.content)

        if previous.content_hash == new_hash:
            return RegistrationResult(
                RegistrationAction.UNCHANGED,
                document_id,
                previous.version_id,
                "Content hash unchanged",
            )

        version = DocumentVersion(
            version_id=str(uuid.uuid4()),
            document_id=document_id,
            version_number=previous.version_number + 1,
            content_hash=new_hash,
            size_bytes=len(source.content),
            created_at=datetime.now(timezone.utc),
        )

        self.registry.append_version(document_id, version)

        return RegistrationResult(
            RegistrationAction.UPDATE,
            document_id,
            version.version_id,
            "New content version",
        )

In [9]:
registry = InMemoryDocumentRegistry()
registration = DocumentRegistrationService(registry)

first = registration.create(source)
print(first)

same = registration.update(first.document_id, source)
print(same)

assert same.action == RegistrationAction.UNCHANGED
assert same.version_id == first.version_id

# Independent create with identical bytes
second = registration.create(source)

assert second.document_id != first.document_id
assert registry.get_active_version(
    second.document_id
).content_hash == registry.get_active_version(
    first.document_id
).content_hash

# Modified content for the existing document
updated_source = SourceFile(
    source_uri=source.source_uri,
    filename=source.filename,
    content=source.content + b"\nRetrieval supports metadata filters.",
)

updated = registration.update(
    first.document_id,
    updated_source,
)

assert updated.action == RegistrationAction.UPDATE
assert updated.document_id == first.document_id
assert len(registry.list_versions(first.document_id)) == 2

RegistrationResult(action=<RegistrationAction.CREATE: 'create'>, document_id='b2af4e98-e56a-4953-b7a7-df7c6cfdd349', version_id='16cdbd50-4c43-44fb-b316-5631a837d162', reason='New logical document')
RegistrationResult(action=<RegistrationAction.UNCHANGED: 'unchanged'>, document_id='b2af4e98-e56a-4953-b7a7-df7c6cfdd349', version_id='16cdbd50-4c43-44fb-b316-5631a837d162', reason='Content hash unchanged')


In [10]:
@dataclass(frozen=True)
class ParsedDocument:
    document_id: str
    version_id: str
    text: str
    metadata: dict


class TextParser:
    def parse(
        self,
        source: SourceFile,
        document_id: str,
        version_id: str,
    ) -> ParsedDocument:

        text = source.content.decode("utf-8")

        return ParsedDocument(
            document_id=document_id,
            version_id=version_id,
            text=text,
            metadata={
                "source_uri": source.source_uri,
                "filename": source.filename,
            },
        )


parser = TextParser()

parsed = parser.parse(
    source,
    first.document_id,
    first.version_id,
)

print(parsed.text)

KnowledgeForge is a document retrieval system.
    It uses Python for ingestion.
    Documents are chunked and embedded.
    Chroma stores the vectors.
    Ollama Qwen generates answers using retrieved context.


In [11]:
@dataclass(frozen=True)
class Chunk:
    chunk_id: str
    document_id: str
    version_id: str
    position: int
    text: str
    metadata: dict

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


class RecursiveChunker:
    def __init__(
        self,
        chunk_size: int = 500,
        chunk_overlap: int = 100,
    ):
        if chunk_size <= 0 or not 0 <= chunk_overlap < chunk_size:
            raise ValueError("Invalid chunking configuration")

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    def split(self, document: ParsedDocument) -> list[Chunk]:
        texts = self.splitter.split_text(document.text)

        return [
            Chunk(
                chunk_id=f"{document.version_id}:{position}",
                document_id=document.document_id,
                version_id=document.version_id,
                position=position,
                text=text,
                metadata=dict(document.metadata),
            )
            for position, text in enumerate(texts)
        ]


chunker = RecursiveChunker(chunk_size=150, chunk_overlap=30)

chunks = chunker.split(parsed)

for chunk in chunks:
    print(chunk.chunk_id, repr(chunk.text))

16cdbd50-4c43-44fb-b316-5631a837d162:0 'KnowledgeForge is a document retrieval system.\n    It uses Python for ingestion.\n    Documents are chunked and embedded.'
16cdbd50-4c43-44fb-b316-5631a837d162:1 'Chroma stores the vectors.\n    Ollama Qwen generates answers using retrieved context.'


In [13]:
from sentence_transformers import SentenceTransformer

class LocalEmbeddingProvider:
    def __init__(self, model_name:str="sentence-transformers/all-MiniLM-L6-v2"):
        self.model=SentenceTransformer(model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.model.encode(
            texts,
            normalize_embeddings=True,
        ).tolist()

    def embed_query(self, query: str) -> list[float]:
        return self.model.encode(
            query,
            normalize_embeddings=True,
        ).tolist()
embedder = LocalEmbeddingProvider()

/Users/chinmayhegde/Workspace/Learning/rag-learning-hub/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1101 07:24:45.517000 51339 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11443.45it/s]


In [14]:
vectors = embedder.embed_documents(
    [chunk.text for chunk in chunks]
)

print("Number of chunks:", len(chunks))
print("Number of embeddings:", len(vectors))
print("Embedding dimension:", len(vectors[0]))
print("First five values:", vectors[0][:5])

assert len(vectors) == len(chunks)
assert len(vectors[0]) == 384

Number of chunks: 2
Number of embeddings: 2
Embedding dimension: 384
First five values: [-0.015225795097649097, 0.016211338341236115, -0.10411326587200165, 0.056287653744220734, 0.04986152425408363]


In [15]:
import chromadb


class ChromaVectorStore:
    def __init__(self, collection_name: str = "knowledgeforge"):
        self.client = chromadb.EphemeralClient()
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )

    def upsert(self, chunks: list[Chunk], vectors: list[list[float]]):
        if len(chunks) != len(vectors):
            raise ValueError("Chunk and embedding counts differ")

        if not chunks:
            return

        self.collection.upsert(
            ids=[c.chunk_id for c in chunks],
            documents=[c.text for c in chunks],
            embeddings=vectors,
            metadatas=[
                {
                    **c.metadata,
                    "document_id": c.document_id,
                    "version_id": c.version_id,
                    "position": c.position,
                }
                for c in chunks
            ],
        )

    def search(
        self,
        query_vector: list[float],
        top_k: int = 3,
        where: dict | None = None,
    ):
        if top_k <= 0:
            raise ValueError("top_k must be positive")

        return self.collection.query(
            query_embeddings=[query_vector],
            n_results=top_k,
            where=where,
            include=["documents", "metadatas", "distances"],
        )

    def delete_version(self, version_id: str):
        self.collection.delete(where={"version_id": version_id})


store = ChromaVectorStore()
store.upsert(chunks, vectors)

print("Indexed chunks:", store.collection.count())

Indexed chunks: 2


In [16]:
class InMemoryPublicationRegistry:
    def __init__(self):
        self._published = {}

    def publish(self, document_id: str, version_id: str):
        self._published[document_id] = version_id

    def get(self, document_id: str) -> str | None:
        return self._published.get(document_id)


publication = InMemoryPublicationRegistry()

# Publish V1 because it has been indexed successfully.
publication.publish(first.document_id, first.version_id)

print("Latest registered:", registry.get(first.document_id).active_version_id)
print("Published version:", publication.get(first.document_id))

Latest registered: 2f97a658-345c-481a-8cbe-0ccc53c73f79
Published version: 16cdbd50-4c43-44fb-b316-5631a837d162


In [17]:
class VectorRetriever:
    def __init__(
        self,
        embedder: LocalEmbeddingProvider,
        store: ChromaVectorStore,
        publication: InMemoryPublicationRegistry,
    ):
        self.embedder = embedder
        self.store = store
        self.publication = publication

    def retrieve(
        self,
        question: str,
        document_id: str,
        top_k: int = 3,
    ) -> list[dict]:

        published_version = self.publication.get(document_id)

        if published_version is None:
            return []

        query_vector = self.embedder.embed_query(question)

        results = self.store.search(
            query_vector=query_vector,
            top_k=top_k,
            where={
                "$and": [
                    {"document_id": document_id},
                    {"version_id": published_version},
                ]
            },
        )

        return [
            {
                "text": text,
                "metadata": metadata,
                "distance": distance,
            }
            for text, metadata, distance in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
        ]


retriever = VectorRetriever(embedder, store, publication)

In [18]:
question = "Which database stores the vectors?"

results = retriever.retrieve(
    question,
    document_id=first.document_id,
    top_k=3,
)

for result in results:
    print("Distance:", result["distance"])
    print("Text:", result["text"])
    print("Source:", result["metadata"]["filename"])
    print("---")

Distance: 0.43991851806640625
Text: Chroma stores the vectors.
    Ollama Qwen generates answers using retrieved context.
Source: architecture.txt
---
Distance: 0.72057044506073
Text: KnowledgeForge is a document retrieval system.
    It uses Python for ingestion.
    Documents are chunked and embedded.
Source: architecture.txt
---


In [19]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate


llm = ChatOllama(
    model="qwen3:8b",
    temperature=0,
)

llm.invoke("hi")

AIMessage(content='Hello! 😊 How can I assist you today? Let me know if you need help with anything!', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-11-01T01:55:09.055Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3473716250, 'load_duration': 6662834, 'prompt_eval_count': 11, 'prompt_eval_duration': 50694000, 'eval_count': 144, 'eval_duration': 3405448000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a194d1-46eb-702a-b710-880ab2ca4227-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 144, 'total_tokens': 155})

In [20]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a document question-answering assistant.

        Answer using only the provided context.
        If the answer is not present in the context,
        say that you don't know based on the documents.

        Do not invent facts.

        Context:
        {context}
        """
    ),
    ("human", "{question}"),
])

In [21]:

@dataclass(frozen=True)
class RAGResponse:
    answer: str
    sources: list[dict]


class RAGService:
    def __init__(self, retriever, llm, prompt):
        self.retriever = retriever
        self.chain = prompt | llm

    def ask(
        self,
        question: str,
        document_id: str,
    ) -> RAGResponse:

        retrieved = self.retriever.retrieve(
            question,
            document_id=document_id,
            top_k=3,
        )

        if not retrieved:
            return RAGResponse(
                answer="No relevant document content is available.",
                sources=[],
            )

        context = "\n\n".join(
            f"[Source {i + 1}]\n{item['text']}"
            for i, item in enumerate(retrieved)
        )

        response = self.chain.invoke({
            "question": question,
            "context": context,
        })

        return RAGResponse(
            answer=response.content,
            sources=retrieved,
        )


rag = RAGService(retriever, llm, rag_prompt)

In [22]:
response = rag.ask(
    "Which database stores the vectors?",
    document_id=first.document_id,
)

print("ANSWER:")
print(response.answer)

print("\nSOURCES:")
for source_item in response.sources:
    print(source_item["metadata"]["filename"])
    print(source_item["text"])

ANSWER:
The database that stores the vectors is **Chroma**, as stated in Source 1.

SOURCES:
architecture.txt
Chroma stores the vectors.
    Ollama Qwen generates answers using retrieved context.
architecture.txt
KnowledgeForge is a document retrieval system.
    It uses Python for ingestion.
    Documents are chunked and embedded.
